In [1]:
import json

import numpy as np
import pandas as pd
import geopandas as gpd
from tqdm import tqdm

from shapely.geometry import Polygon

Load census variables and parse for relevant information

In [2]:
TARIFF_IMPACT_FILE = '../raw/tariffs/ada_tariff_counts_percents_9_29_2026.xlsx'

# Create mapping for characteristic IDs to column names
characteristic_mapping = {
    1: 'CHAR_POP21'  # Population, 2021
}

df_cen_cma_data = pd.read_csv('../../data/census/98-401-X2021012_English_CSV_data.csv', encoding='latin')
df_cen_cma_data = df_cen_cma_data[['DGUID', 'GEO_LEVEL', 'GEO_NAME', 'CHARACTERISTIC_ID', 'C1_COUNT_TOTAL']]

# Filter for only the characteristics we want
df_cen_cma_data = df_cen_cma_data[df_cen_cma_data['CHARACTERISTIC_ID'].isin(characteristic_mapping.keys())]

# Map characteristic IDs to column names
df_cen_cma_data['CHARACTERISTIC_COLUMN'] = df_cen_cma_data['CHARACTERISTIC_ID'].map(characteristic_mapping)

# Pivot to create separate columns for each characteristic
df_cen_cma_data = df_cen_cma_data.pivot_table(
    index=['DGUID', 'GEO_LEVEL', 'GEO_NAME'], 
    columns='CHARACTERISTIC_COLUMN', 
    values='C1_COUNT_TOTAL',
    aggfunc='first'
).reset_index()

# Flatten column names
df_cen_cma_data.columns.name = None

In [3]:
df_ada_cma_rel = pd.read_csv('../../data/census/ada_cma_relation.csv')
df_ada_cma_rel = df_ada_cma_rel.rename(columns={'CMADGUID_RMRIDUGD': 'CMADGUID', 'ADADGUID_ADAIDUGD': 'ADADGUID'})

In [4]:
# lcma000b21a_e is a folder containing the shapefile components of census metropolitan areas of Canada 2021
gdf_cma = gpd.read_file('../../data/census/lcma000b21a_e')
gdf_cma = gdf_cma[['CMAUID', 'DGUID', 'CMANAME', 'CMATYPE', 'PRUID', 'geometry']]

# Merge duplicates on DGUID: first row's attrs + unioned geometry
gdf_cma = (
    gdf_cma
    .groupby('DGUID', as_index=False)
    .agg({
        'CMAUID': 'first',
        'CMANAME': 'first',
        'CMATYPE': 'first',
        'PRUID': 'first',
        'geometry': lambda x: x.unary_union
    })
)

# Remove anything inside parentheses (and the parentheses themselves)
gdf_cma['CMANAME'] = gdf_cma['CMANAME'].str.replace(r"\s*\(.*?\)", "", regex=True).str.strip()

gdf_cma = gpd.GeoDataFrame(gdf_cma, geometry='geometry')
if gdf_cma.crs is None:
    gdf_cma.set_crs("EPSG:3347", inplace=True)


print(gdf_cma.CMATYPE.value_counts())

CMATYPE
D    102
B     41
K      9
Name: count, dtype: int64


Load tariff information and join to CMAs

In [6]:
# Load ADA-level tariff counts and percents
df_tariffs_ada_count = pd.read_excel(TARIFF_IMPACT_FILE, sheet_name='Counts')#.drop(columns=['geometry'])
df_tariffs_ada_pct = pd.read_excel(TARIFF_IMPACT_FILE, sheet_name='Percents')

In [7]:
df_tariffs_count_filtered = (
    df_tariffs_ada_count
    # .drop(columns=['geometry'])
    .merge(df_ada_cma_rel, on='ADADGUID', how='left')
    .dropna(subset=['CMADGUID'])
)

# Group by CMADGUID and sum all tariff columns
tariff_columns_count = [col for col in df_tariffs_count_filtered.columns if col not in ['ADADGUID', 'CMADGUID']]
df_tariffs_cma_count = df_tariffs_count_filtered.groupby('CMADGUID')[tariff_columns_count].sum().reset_index()

print(f"Shape of CMA tariffs (counts) dataframe: {df_tariffs_cma_count.shape}")
df_tariffs_cma_count.head()

Shape of CMA tariffs (counts) dataframe: (152, 43)


,CMADGUID,All_Businesses,All_Employees,Auto_B,Auto_E,Alum_B,Alum_E,Steel_B,Steel_E,Cop_B,...,S338_Tot_B,S338_Tot_E,CUSMA_B,CUSMA_E,Total_B,Total_E,ScenBefore_B,ScenBefore_E,ScenAfter_B,ScenAfter_E
0,2021S0503001,6797,117264,7,41,17,105,14,95,2,...,79,919,118,2046,118,2046,118,2030,118,2046
1,2021S0503205,13225,240984,35,708,90,2087,57,1222,10,...,307,4582,404,6417,421,7251,414,6513,421,7251
2,2021S0503305,4837,87648,15,317,37,798,34,685,5,...,124,3033,171,3604,191,4346,186,3796,191,4346
3,2021S0503310,3515,65122,5,153,17,399,13,1204,1,...,106,3519,136,2877,142,3904,142,3582,142,3904
4,2021S0503320,3311,55666,9,172,26,360,15,262,0,...,81,810,126,1667,132,1833,132,1730,132,1833


In [8]:
# Melt percents and counts to long format, extract base and numeric suffix
def melt_tariffs(df, value_name, suffix_map=None):
    df_long = df.melt(id_vars=['ADADGUID'], var_name='tariff', value_name=value_name)
    df_long['base'] = df_long['tariff'].str.replace(r'_(1|2|3|B|E|C)$', '', regex=True)
    df_long['suffix'] = df_long['tariff'].str.extract(r'_([1-3BEC])$')[0]
    if suffix_map:
        df_long['suffix'] = df_long['suffix'].map(suffix_map)
    return df_long

suffix_map_counts = {'B': '1', 'E': '2', 'C': '3'}

df_pct_long = melt_tariffs(df_tariffs_ada_pct, 'percent')
df_count_long = melt_tariffs(df_tariffs_ada_count, 'count', suffix_map=suffix_map_counts)

# Merge percents and counts, add CMA IDs
df_merge = (
    df_pct_long
    .merge(df_count_long, on=['ADADGUID', 'base', 'suffix'], how='left')
    .merge(df_ada_cma_rel, on='ADADGUID', how='left')
    .dropna(subset=['CMADGUID'])
)

# Compute weighted percent and aggregate per CMA
df_cma = (
    df_merge.assign(weighted=lambda x: x['percent'] * x['count'])
    .groupby(['CMADGUID', 'base', 'suffix'], observed=True)
    .agg(total_weighted=('weighted', 'sum'), total_count=('count', 'sum'))
    .reset_index()
)
df_cma['cma_percent'] = (df_cma['total_weighted'] / df_cma['total_count']) * 100
df_cma.loc[df_cma['total_count'] == 0, 'cma_percent'] = np.nan

# Pivot to wide format
df_cma['colname'] = df_cma['base'] + '_' + df_cma['suffix']
df_tariffs_cma_pct = df_cma.pivot(index='CMADGUID', columns='colname', values='cma_percent').reset_index()

print(f"Shape of CMA tariffs (percents) dataframe: {df_tariffs_cma_pct.shape}")
df_tariffs_cma_pct.head()

Shape of CMA tariffs (percents) dataframe: (152, 41)


colname,CMADGUID,Alcohol_1,Alcohol_2,Alum_1,Alum_2,Auto_1,Auto_2,CUSMA_1,CUSMA_2,Cop_1,...,ScenBefore_1,ScenBefore_2,Steel_1,Steel_2,Total_1,Total_2,after September 29_1,after September 29_2,before September 29_1,before September 29_2
0,2021S0503001,0.410149,0.249816,0.569074,0.691774,0.510324,0.208220,2.111118,7.708065,0.261214,...,2.111118,7.512447,0.744515,0.824444,2.111118,7.708065,1.723631,1.642633,1.220461,1.345830
1,2021S0503205,0.813391,0.378418,1.273492,3.284860,0.609318,1.689228,6.063522,6.134122,0.718565,...,6.164088,6.181843,0.966178,1.702845,6.210002,6.692827,4.932417,5.963024,2.284452,4.883807
2,2021S0503305,0.397276,0.694481,1.162877,2.568543,0.663729,0.847195,5.366186,7.667143,0.410068,...,5.893414,8.706873,1.525981,2.509847,6.062326,9.286939,4.522583,8.820106,2.588903,4.349029
3,2021S0503310,0.968335,5.982111,1.065927,2.744047,0.901604,3.344821,10.848149,6.243075,2.380952,...,10.772117,9.166756,1.251197,8.082384,10.772117,9.334061,10.369096,9.231930,2.871069,7.120852
4,2021S0503320,1.246377,0.683107,1.389997,2.018435,0.688483,2.629791,6.504885,16.186146,NaN,...,6.647012,16.268406,1.651715,2.551230,6.647012,15.992116,4.486483,16.907407,2.804928,18.149476


In [9]:
# First, we need to aggregate census data from ADA to CMA level
df_cen_ada = df_cen_cma_data.rename(columns={'DGUID': 'ADADGUID'})

# Merge census ADA data with ADA-CMA relationship
df_cen_with_cma = df_cen_ada.merge(df_ada_cma_rel, on='ADADGUID', how='left')

# Aggregate population to CMA level
df_cen_cma = df_cen_with_cma.groupby('CMADGUID').agg({
    'GEO_NAME': 'first',  # replaced below with the proper CMA name
    'CHAR_POP21': 'sum'
}).reset_index()

# Merge with GDF to get proper names AND the CMA/CA type flag
gdf_cma_names = (gdf_cma[['DGUID', 'CMANAME', 'CMATYPE']]
                 .rename(columns={'DGUID': 'CMADGUID', 'CMANAME': 'GEO_NAME'}))
df_cen_cma = df_cen_cma.drop(columns=['GEO_NAME']).merge(gdf_cma_names, on='CMADGUID', how='left')

# Derive GEO_LEVEL from CMATYPE. Never hardcode this: 'B' is a census
# metropolitan area, 'D' and 'K' are census agglomerations.
CMA_CODE = 'B'
df_cen_cma['GEO_LEVEL'] = np.where(df_cen_cma.CMATYPE == CMA_CODE,
                                   'Census metropolitan area',
                                   'Census agglomeration')

assert (df_cen_cma.GEO_LEVEL == 'Census metropolitan area').sum() == 41, \
    df_cen_cma.GEO_LEVEL.value_counts()

# Now merge with tariff data
df_final_counts = df_cen_cma.merge(df_tariffs_cma_count, on='CMADGUID', how='inner')
df_final_percents = df_cen_cma.merge(df_tariffs_cma_pct, on='CMADGUID', how='inner')

df_final_counts = df_final_counts[df_final_counts['CMATYPE'] == 'B']
df_final_percents = df_final_percents[df_final_percents['CMATYPE'] == 'B']

print(f"Shape of final counts dataframe: {df_final_counts.shape}")
print(f"Shape of final percents dataframe: {df_final_percents.shape}")
print(df_final_counts.GEO_LEVEL.value_counts().to_string())

Shape of final counts dataframe: (41, 47)
Shape of final percents dataframe: (41, 45)
GEO_LEVEL
Census metropolitan area    41


In [10]:
# Prepare geometry data
gdf_cma_geom = gpd.GeoDataFrame(
    gdf_cma[['DGUID', 'geometry']].rename(columns={'DGUID': 'CMADGUID'}),
    geometry='geometry',
    crs=gdf_cma.crs
)

# ---- COUNTS ----
gdf_final_counts_full = gpd.GeoDataFrame(
    df_final_counts.merge(gdf_cma_geom, on='CMADGUID', how='inner'),
    geometry='geometry',
    crs=gdf_cma.crs
)

# Centroids for counts
gdf_cma_centroids = gdf_cma_geom.copy()
gdf_cma_centroids['geometry'] = gdf_cma_centroids.geometry.centroid
gdf_cma_centroids = gdf_cma_centroids.to_crs('EPSG:4326')

gdf_final_counts_centroids = gpd.GeoDataFrame(
    df_final_counts.merge(gdf_cma_centroids, on='CMADGUID', how='inner'),
    geometry='geometry',
    crs='EPSG:4326'
)

# Save counts
gdf_final_counts_full.to_file('../../data/cma/cma_tariffs_counts_full_geometry.gpkg', driver='GPKG')

# Save counts centroids as CSV: convert geometry to WKT only for the CSV copy
df_counts_centroids_csv = gdf_final_counts_centroids.copy()
df_counts_centroids_csv['geometry'] = df_counts_centroids_csv.geometry.apply(lambda geom: geom.wkt)
df_counts_centroids_csv.to_csv('../../data/cma/cma_tariffs_counts_centroids.csv', index=False)

# ---- PERCENTS ----
gdf_final_percents_full = gpd.GeoDataFrame(
    df_final_percents.merge(gdf_cma_geom, on='CMADGUID', how='inner'),
    geometry='geometry',
    crs=gdf_cma.crs
)

gdf_final_percents_centroids = gpd.GeoDataFrame(
    df_final_percents.merge(gdf_cma_centroids, on='CMADGUID', how='inner'),
    geometry='geometry',
    crs='EPSG:4326'
)

# Save percents
gdf_final_percents_full.to_file('../../data/cma/cma_tariffs_percents_full_geometry.gpkg', driver='GPKG')

# Save percents centroids as CSV: convert geometry to WKT only for the CSV copy
df_percents_centroids_csv = gdf_final_percents_centroids.copy()
df_percents_centroids_csv['geometry'] = df_percents_centroids_csv.geometry.apply(lambda geom: geom.wkt)
df_percents_centroids_csv.to_csv('../../data/cma/cma_tariffs_percents_centroids.csv', index=False)

# Print shapes
print(f"Counts full geometry shape: {gdf_final_counts_full.shape}")
print(f"Percents full geometry shape: {gdf_final_percents_full.shape}")
print(f"Counts centroids shape: {gdf_final_counts_centroids.shape}")
print(f"Percents centroids shape: {gdf_final_percents_centroids.shape}")


C:\Users\yihoi\AppData\Local\Temp\ipykernel_18784\3454657186.py:31: UserWarning: Geometry column does not contain geometry.
  df_counts_centroids_csv['geometry'] = df_counts_centroids_csv.geometry.apply(lambda geom: geom.wkt)


Counts full geometry shape: (41, 48)
Percents full geometry shape: (41, 46)
Counts centroids shape: (41, 48)
Percents centroids shape: (41, 46)


C:\Users\yihoi\AppData\Local\Temp\ipykernel_18784\3454657186.py:52: UserWarning: Geometry column does not contain geometry.
  df_percents_centroids_csv['geometry'] = df_percents_centroids_csv.geometry.apply(lambda geom: geom.wkt)


In [11]:
counts_path = '../../data/cma/cma_tariffs_counts_centroids.csv'
percents_path = '../../data/cma/cma_tariffs_percents_centroids.csv'

counts_json_path = '../../data/cma/cma_tariffs_counts_centroids.json'
percents_json_path = '../../data/cma/cma_tariffs_percents_centroids.json'

def csv_to_json_records(csv_path):
    df = pd.read_csv(csv_path)

    # Replace all NaN, NaT, and inf values with None so they become valid JSON null
    df = df.replace({np.nan: None, np.inf: None, -np.inf: None})

    return df.to_dict(orient='records')

counts_records = csv_to_json_records(counts_path)
percents_records = csv_to_json_records(percents_path)

with open(counts_json_path, 'w', encoding='utf-8') as f:
    json.dump(counts_records, f, ensure_ascii=False, indent=4)
with open(percents_json_path, 'w', encoding='utf-8') as f:
    json.dump(percents_records, f, ensure_ascii=False, indent=4)

print(f'Saved counts JSON: {counts_json_path} (rows={len(counts_records)})')
print(f'Saved percents JSON: {percents_json_path} (rows={len(percents_records)})')


Saved counts JSON: ../../data/cma/cma_tariffs_counts_centroids.json (rows=41)
Saved percents JSON: ../../data/cma/cma_tariffs_percents_centroids.json (rows=41)


In [12]:
df_cma_pcts = pd.read_csv(percents_path)
df_cma_pcts = df_cma_pcts[df_cma_pcts['GEO_LEVEL'] == 'Census metropolitan area']
df_cma_pcts.describe(percentiles=[0.2, 0.4, 0.6, 0.8]).round(2).to_csv('../../data/cma/cma_tariffs_percents_variance.csv')

In [13]:
# old = pd.read_excel('../raw/tariff-impacts-ada-data.xlsx', sheet_name='Percents')
# new = pd.read_excel('../raw/tariff-impacts-ada-data_9_7_2026.xlsx', sheet_name='Percents')
# m = old.merge(new, on='ADADGUID', suffixes=('_old','_new'))
# print((m['CUSMA_3_new'] - m['CUSMA_3_old']).describe())
# print((m['Total_3_new'] - m['Total_3_old']).describe())